In [0]:
import sys
sys.path.append("../python")

In [0]:
%pip install -r ../requirements.txt

In [0]:
%pip install --upgrade numpy tensorflow

In [0]:
%restart_python

In [0]:
from create_qa import GenerateQAContent

In [0]:
import json
from tqdm import tqdm
#Not used for now, didn't seem to improve much the results, and needed to be adapted to the next classes to process the raw outputs
from generated_prompt import prompt_template 
from transformers import pipeline, AutoModelForCausalLM, BitsAndBytesConfig, AutoTokenizer


In [0]:
from openai import OpenAI
import os

# How to get your Databricks token: https://docs.databricks.com/en/dev-tools/auth/pat.html
#DATABRICKS_TOKEN = os.environ.get('DATABRICKS_TOKEN')
# Alternatively in a Databricks notebook you can use this:
DATABRICKS_TOKEN = dbutils.notebook.entry_point.getDbutils().notebook().getContext().apiToken().get()

client = OpenAI(
  api_key=DATABRICKS_TOKEN,
  base_url="https://adb-4424763777139025.5.azuredatabricks.net/serving-endpoints"
)
#

In [0]:
chat_completion = client.chat.completions.create(
  messages=[
  {
    "role": "system",
    "content": "You are an AI assistant"
  },
  {
    "role": "user",
    "content": "Tell me about Large Language Models"
  }
  ],
  model="databricks-meta-llama-3-1-8b-instruct",
  max_tokens=256
)

print(chat_completion.choices[0].message.content)

In [0]:
from databricks.sdk import WorkspaceClient
from tqdm import tqdm
import json
import time

# Connect to workspace
w = WorkspaceClient()

In [0]:
MODEL_NAME = "databricks-meta-llama-3-1-8b-instruct"
model_id = "databricks-meta-llama-3-1-8b-instruct"

file_name = "../data/rebuilding_milo_chunks_docling_max_tokens128_min_tokens50_meta_llama3p18B.txt" 

# Example dataset to generate content from
samples = [
    "Explain how posture affects back pain in physiotherapy.",
    "Describe 3 safe exercises for shoulder rehabilitation.",
    "What are the common causes of knee instability?",
    "How can breathing techniques help in physical recovery?"
]


In [0]:

def get_tokenizer(model_id):
    tokenizer = AutoTokenizer.from_pretrained(model_id)
    tokenizer.pad_token = tokenizer.eos_token
    # tokenizer = AutoTokenizer.from_pretrained(self.model_id, padding_side="left", 
    #                                           max_tokens=256)
    #eos -> end of string token is the pad token
    tokenizer.pad_token = tokenizer.eos_token
    return tokenizer

def llama_token_len(self, text):
    return len(self.get_tokenizer().encode(text))

In [0]:
def get_prompt(text):
    messages = [
        {
            "role": "system",
            "content": (
                "You are a concise and helpful medical tutor. "
                "Based on the provided text, generate a JSON object with exactly ONE question (as 'instruction') and ONE answer (as 'output').\n\n"
                "- The content must relate to health, exercise, sports, fitness, or physiotherapy.\n"
                "- Do not include multiple questions or answers.\n"
                "- Do not repeat the instruction in the output.\n"
                "- Keep the output brief and informative.\n"
                "- If the text is not relevant, return: {\"instruction\": \"NULL\", \"output\": \"NULL\"}\n\n"
                "- Respond ONLY with the JSON object. Do NOT include any explanation or commentary."
            ),
        },
        {
            "role": "user",
            "content": text.strip()
        },
    ]
    return messages
    #tokenizer = get_tokenizer(model_id)
    #return tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)

In [0]:
def get_text():
    # with open(self.file_name, "r") as f:
    #     llama_chunks = json.load(f)
    with open(file_name, 'r') as f:
        llama_chunks = f.readlines()
    return llama_chunks 
    

def get_samples(n_chunks_intervals=None, n_repetitions=3, save_json=False, batch_size=16):
    text_chunks = get_text()
    raw_outputs = []
    samples = text_chunks[:] if n_chunks_intervals == None else text_chunks[n_chunks_intervals[0]:n_chunks_intervals[1]]
    return samples


In [0]:
import mlflow

mlflow.tracing.disable()


In [0]:
results = []
for sample in samples[30:31]:
    prompt = get_prompt(sample)
    chat_completion = client.chat.completions.create(
    messages=prompt,
    model=MODEL_NAME,
    max_tokens=256
    )

    try:
        content = chat_completion.choices[0].message.content

        try:
            parsed = json.loads(content)
            instruction = parsed.get("instruction", "")
            output = parsed.get("output", "")
        except json.JSONDecodeError:
            # Fallback in case model returns plain text
            instruction, output = None, content

        results.append({
            "prompt": prompt,
            "instruction": instruction,
            "output": output
        })

        print("*" * 20)
        print(f"Instruction: {instruction}")
        print(f"Output: {output}")

    except Exception as e:
        print("Error parsing completion:", e)
        results.append({
            "prompt": prompt,
            "output": str(chat_completion)
        })

In [0]:
len(results)

In [0]:


results = []

for sample in samples[:100]:
    prompt = get_prompt(sample)

    chat_completion = client.chat.completions.create(
        messages=prompt,
        model=MODEL_NAME,
        max_tokens=256,
        temperature=0.75,         # creative diversity
        top_p=0.65,
        n=3,                     # ✅ generate 3 completions per prompt
    )

    for choice in chat_completion.choices:
        content = choice.message.content

        try:
            parsed = json.loads(content)
            instruction = parsed.get("instruction", "")
            output = parsed.get("output", "")
        except json.JSONDecodeError:
            instruction, output = None, content

        results.append({
            "prompt": prompt,
            "instruction": instruction,
            "output": output
        })

json_output_name= f"test_16-10-25"
with open(f"{json_output_name}.json", "w") as f:
    json.dump(results, f)



In [0]:
import json
from tqdm import tqdm
import time

results = []

batch_size = 5  # number of samples per batch
n_reps = 3      # number of completions per sample
temperature = 0.75
top_p = 0.65

# Helper to chunk a list
def chunk_list(lst, size):
    for i in range(0, len(lst), size):
        yield lst[i:i+size]

for batch in tqdm(chunk_list(samples[100:500], batch_size)):
    # Build batch prompts
    batch_prompts = [get_prompt(sample) for sample in batch]

    # Iterate over each sample in the batch
    for prompt in batch_prompts:
        for _ in range(n_reps):
            try:
                chat_completion = client.chat.completions.create(
                    messages=prompt,
                    model=MODEL_NAME,
                    max_tokens=256,
                    temperature=temperature,
                    top_p=top_p,
                    n=1  # one completion per API call; we repeat for n_reps
                )
                content = chat_completion.choices[0].message.content

                try:
                    parsed = json.loads(content)
                    instruction = parsed.get("instruction", "")
                    output = parsed.get("output", "")
                except json.JSONDecodeError:
                    instruction, output = None, content

                results.append({
                    "prompt": prompt,
                    "instruction": instruction,
                    "output": output
                })

                # Optional: short sleep to avoid hitting rate limits
                time.sleep(0.5)

            except Exception as e:
                print(f"Error generating completion for prompt: {prompt}")
                print(e)

json_output_name= f"test_16-10-25-100to500"
with open(f"{json_output_name}.json", "w") as f:
    json.dump(results, f)



In [0]:
results[0]

In [0]:
results

In [0]:
json_output_name= f"test_16-10-25"
with open(f"{json_output_name}.json", "w") as f:
    json.dump(results, f)


In [0]:
for result in results:
    print(f"content: {result['prompt'][1]['content']}")
    print(f"instruction: {result['instruction']}")
    
    print(f"output: {result['output']}")
    print("*" * 20)

In [0]:
results = []
for sample in samples[30:32]:
   
    chat_completion = client.chat.completions.create(
    messages=[
    {
        "role": "system",
        "content": "You are an AI assistant"
    },
    {
        "role": "user",
        "content": sample
    }
    ],
    model="databricks-meta-llama-3-1-8b-instruct",
    max_tokens=256
    )
    try:
        output_text = chat_completion["predictions"][0]["candidates"][0]["message"]["content"]
    except Exception:
        output_text = str(chat_completion)

    results.append({
        "prompt": prompt_text,
        "output": output_text
    })



In [0]:
results

In [0]:



# Number of generations per sample
n_repetitions = 1
batch_size = 2

results = []

for i in tqdm(range(0, len(samples), batch_size)):
    batch = samples[i:i+batch_size]
    print(f"Processing batch {i} → {i+len(batch)}")

    for prompt_text in batch:
        for _ in range(n_repetitions):
            chat_completion = w.serving_endpoints.query(
                name=MODEL_NAME,
                inputs={
                    "messages": [
                        {"role": "system", "content": "You are an expert physiotherapist assistant."},
                        {"role": "user", "content": prompt_text}
                    ],
                    "max_tokens": 512,
                    "temperature": 0.7,
                    "top_p": 0.9
                }
            )

            # Extract model output text
            try:
                output_text = chat_completion["predictions"][0]["candidates"][0]["message"]["content"]
            except Exception:
                output_text = str(chat_completion)

            results.append({
                "prompt": prompt_text,
                "output": output_text
            })

            # Optional pause to avoid rate limits
            time.sleep(0.5)

"""# Store results as Delta or JSON for further fine-tuning
df = spark.createDataFrame(results)
df.write.format("delta").mode("overwrite").saveAsTable("synthetic_qa_dataset")

print("✅ Synthetic dataset stored in Delta table: synthetic_qa_dataset")
"""

In [0]:
file_name = "../data/rebuilding_milo_chunks_docling_max_tokens128_min_tokens50_meta_llama3p18B.txt" 
model_id = "databricks-meta-llama-3-1-8b-instruct"

def get_text(self):
    # with open(self.file_name, "r") as f:
    #     llama_chunks = json.load(f)
    with open(file_name, 'r') as f:
        llama_chunks = f.readlines()
    return llama_chunks 
    

In [0]:
file_name = "../data/rebuilding_milo_chunks_docling_max_tokens128_min_tokens50_meta_llama3p18B.txt" 
model_id = "meta-llama/Llama-3.1-8B-Instruct" #TinyLlama/TinyLlama-1.1B-Chat-v1.0"
save_json=False

#For all chunks set to None
n_chunks_intervals=[30,31] 
n_repetitions = 1

gc = GenerateQAContent(file_name, model_id)
text = gc.get_text()
raw_outputs = gc.generate_content(n_chunks_intervals=n_chunks_intervals, n_repetitions=n_repetitions, 
                                    save_json=save_json, batch_size=16)
#print(raw_outputs)